# Capstone: design a browser-safe geospatial investigation

The capstone asks learners to combine at least one real boundary layer, one point layer, one external tile or API layer, and one explicit uncertainty note.

The starter code below builds a modular map function. Learners can swap datasets while preserving a clean map architecture.

**Reflection questions:** What is the decision this map supports? Which layer is most uncertain? Which step would need a server-side GIS stack rather than Pyodide? What would you remove to make the map clearer?

In [ ]:
# Pyodide/JupyterLite bootstrap: install only pure-Python packages used in this notebook.
import sys, importlib
try:
    import micropip
except Exception:
    micropip = None

async def ensure_packages(packages):
    for pkg, import_name in packages:
        try:
            importlib.import_module(import_name)
        except Exception:
            if micropip is None:
                raise RuntimeError(f'{pkg} is not installed and micropip is unavailable.')
            await micropip.install(pkg)

await ensure_packages([('pandas','pandas'), ('folium','folium'), ('branca','branca'), ('plotly','plotly')])


In [ ]:
from pathlib import Path
import json, math, statistics
import pandas as pd
import folium
from folium.plugins import MarkerCluster, HeatMap, TimestampedGeoJson, MiniMap, Fullscreen, MeasureControl

DATA = Path('../data')

def load_json(name):
    return json.loads((DATA / name).read_text(encoding='utf-8'))

def load_csv(name):
    return pd.read_csv(DATA / name)

def add_standard_controls(m):
    MiniMap(toggle_display=True).add_to(m)
    Fullscreen().add_to(m)
    MeasureControl(primary_length_unit='kilometers').add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    return m

def color_scale(values, colors=('green','orange','red')):
    vals = list(values)
    lo, hi = min(vals), max(vals)
    def pick(v):
        if hi == lo:
            return colors[1]
        t = (v - lo) / (hi - lo)
        return colors[0] if t < .33 else colors[1] if t < .66 else colors[2]
    return pick


In [ ]:
def build_capstone(boundary_name='montreal_districts.geojson', point_name='health_facilities_training_points.csv'):
    boundaries = load_json(boundary_name)
    points = load_csv(point_name)
    m = folium.Map(location=[45.52,-73.60], zoom_start=10, tiles='CartoDB positron')
    folium.GeoJson(boundaries, name='Real boundary layer', tooltip=folium.GeoJsonTooltip(fields=['district'])).add_to(m)
    cluster = MarkerCluster(name='Point evidence').add_to(m)
    for _, r in points[points.city.eq('Montreal')].iterrows():
        folium.Marker([r.lat,r.lon], tooltip=r['name']).add_to(cluster)
    folium.plugins.Draw(export=True, filename='learner_annotations.geojson').add_to(m)
    add_standard_controls(m)
    return m

build_capstone()

In [ ]:
# Map audit checklist as data: keep this with every learner map.
audit = pd.DataFrame([
    {'item':'Boundary source','answer':'montreal_districts.geojson; real electoral districts from Plotly example data'},
    {'item':'Point source','answer':'health facility training points; replace with healthsites.io export for production'},
    {'item':'Projection','answer':'WGS84 lon/lat displayed in Web Mercator tiles'},
    {'item':'Known limitation','answer':'Straight-line map only; no travel-time or capacity model'},
    {'item':'Audience','answer':'Learners exploring browser-safe geospatial workflows'},
])
audit